# Week 1

In [3]:
import pandas as pd
import sqlite3

# load the two CSV files
orders = pd.read_csv('../dataset/olist_orders_dataset.csv')
customers = pd.read_csv('../dataset/olist_customers_dataset.csv')
products = pd.read_csv('../dataset/olist_products_dataset.csv')

# create an in-memory SQL database and load them in
conn = sqlite3.connect(':memory:')
orders.to_sql('orders', conn, index=False)
customers.to_sql('customers', conn, index=False)
products.to_sql('products', conn, index=False)

print("Ready. Tables loaded.")

Ready. Tables loaded.


In [4]:
pd.read_sql("""
    SELECT customer_city, COUNT(*) as total_customers
    FROM customers
    GROUP BY customer_city
    ORDER BY total_customers DESC
    LIMIT 10
""", conn)

,customer_city,total_customers
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [5]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
#how many orders does every city have
pd.read_sql("""
    SELECT customer_city, COUNT(*) as total 
    FROM orders
    INNER JOIN customers ON orders.customer_id = customers.customer_id
    GROUP BY customer_city
    ORDER BY total DESC
""", conn)

,customer_city,total
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
...,...,...
4114,acucena,1
4115,acari,1
4116,acajutiba,1
4117,abdon batista,1


In [8]:
# Flag each delivered order as late or on time based on estimated delivery date
pd.read_sql("""
    SELECT order_id,
    CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
        THEN 'Late'
        ELSE 'On time'
    END as delivery_status
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
""", conn)

,order_id,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,On time
1,53cdb2fc8bc7dce0b6741e2150273451,On time
2,47770eb9100c2d0c44946d9cf07ec65d,On time
3,949d5b44dbf5de918fe9c16f97b45f8a,On time
4,ad21c59c0840e6cb83a9ceb5573f8159,On time
...,...,...
96471,9c5dedf39a927c1b2549525ed64a053c,On time
96472,63943bddc261676b46f01ca7ac2f7bd8,On time
96473,83c1379a015df1e13d02aae0204711ab,On time
96474,11c177c8e97725db2631073c19f07b62,On time


In [9]:
# Count how many orders was on time or late
pd.read_sql("""
    SELECT delivery_status, COUNT(*) as total
    FROM (
        SELECT CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 'Late'
                ELSE 'On time'
               END as delivery_status
        FROM orders
        WHERE order_delivered_customer_date IS NOT NULL
    )
    GROUP BY delivery_status
""", conn)

,delivery_status,total
0,Late,7827
1,On time,88649


In [10]:
pd.read_sql("""
    SELECT order_id, ROUND(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)) Delivery_time
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
""", conn)

,order_id,Delivery_time
0,e481f51cbdc54678b7cc49136f2d6af7,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,14.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,3.0
...,...,...
96471,9c5dedf39a927c1b2549525ed64a053c,8.0
96472,63943bddc261676b46f01ca7ac2f7bd8,22.0
96473,83c1379a015df1e13d02aae0204711ab,25.0
96474,11c177c8e97725db2631073c19f07b62,17.0


In [11]:
# Get average delivery time based on state
pd.read_sql("""
    SELECT customer_state, ROUND(AVG(Delivery_time)) average_delivery_time
    FROM (
        SELECT customers.customer_state,
               JULIANDAY(orders.order_delivered_customer_date) - JULIANDAY(orders.order_purchase_timestamp) as delivery_time
        FROM orders
        INNER JOIN customers ON orders.customer_id = customers.customer_id
        WHERE orders.order_delivered_customer_date IS NOT NULL
    )
    GROUP BY customer_state
    ORDER BY average_delivery_time ASC
""",conn)

,customer_state,average_delivery_time
0,SP,9.0
1,MG,12.0
2,PR,12.0
3,DF,13.0
4,RJ,15.0
5,RS,15.0
6,SC,15.0
7,ES,16.0
8,GO,16.0
9,MS,16.0


In [12]:
import os
os.listdir('.')

['.git',
 '.ipynb_checkpoints',
 'olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv',
 'README.md',
 'week1_sql_practice.ipynb']

In [29]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [17]:
pd.read_sql("""
    SELECT customer_city, COUNT(*) AS Total
    FROM customers
    GROUP BY customer_city
""", conn)

,customer_city,Total
0,abadia dos dourados,3
1,abadiania,1
2,abaete,12
3,abaetetuba,11
4,abaiara,2
...,...,...
4114,xinguara,9
4115,xique-xique,3
4116,zacarias,2
4117,ze doca,5


In [23]:
# Count which months we have more orders
pd.read_sql("""
    SELECT STRFTIME('%Y-%m', order_purchase_timestamp) Month, COUNT(*) Orders
    FROM orders
    GROUP BY Month
    ORDER BY Orders DESC
""",conn)

,Month,Orders
0,2017-11,7544
1,2018-01,7269
2,2018-03,7211
3,2018-04,6939
4,2018-05,6873
5,2018-02,6728
6,2018-08,6512
7,2018-07,6292
8,2018-06,6167
9,2017-12,5673


In [24]:
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_items.to_sql('order_items', conn, index=False)
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [28]:
# Count which product sells most
pd.read_sql("""
    SELECT product_id, COUNT(*) Total_order
    FROM order_items
    GROUP BY product_id
    ORDER BY Total_order DESC
""",conn)

,product_id,Total_order
0,aca2eb7d00ea1a7b8ebd4e68314663af,527
1,99a4788cb24856965c36a24e339b6058,488
2,422879e10f46682990de24d770e7f83d,484
3,389d119b48cf3043d311335e499d9c6b,392
4,368c6c730842d78016ad823897a372db,388
...,...,...
32946,0011c512eb256aa0dbbb544d8dffcf6e,1
32947,000d9be29b5207b54e86aa1b1ac54872,1
32948,0009406fd7479715e4bef61dd91f2462,1
32949,00088930e925c41fd95ebfe695fd2655,1


In [32]:
# Count which product category sells most
pd.read_sql("""
    SELECT product_category_name, COUNT(*) Total_order
    FROM order_items
    INNER JOIN products ON products.product_id = order_items.product_id
    GROUP BY product_category_name
    ORDER BY Total_order DESC
""",conn)

,product_category_name,Total_order
0,cama_mesa_banho,11115
1,beleza_saude,9670
2,esporte_lazer,8641
3,moveis_decoracao,8334
4,informatica_acessorios,7827
...,...,...
69,la_cuisine,14
70,cds_dvds_musicais,14
71,pc_gamer,9
72,fashion_roupa_infanto_juvenil,8
